In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "/mnt/ai4s/models_storage/llm-models/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:5"
)

print(model)
print(model.config)

`torch_dtype` is deprecated! Use `dtype` instead!


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [ ]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


In [1]:
import pandas as pd

In [3]:
df = pd.read_json("/mnt/ai4s/zhouhaojie/xinli/data_pre/high_score/rewrite_117/PsyDTCorpus_DPO.json")
df.head()

,messages,rejected_response
0,"[{'role': 'system', 'content': '你是一位精通理情行为疗法（R...",说实话，很多人都有过类似的担心，你不是一个人。如果有一天要不要，或许可以试着和信任的人聊一聊...
1,"[{'role': 'system', 'content': '你是一位精通理情行为疗法（R...",嗯，一路保重，愿你慢慢找到属于自己的安稳与从容。
2,"[{'role': 'system', 'content': '你是一位精通理情行为疗法（R...",是啊，建立信任和连接确实需要时间，但每一步都算数。你已经愿意面对自己的害怕，这自带很勇敢了。...
3,"[{'role': 'system', 'content': '你是一位精通理情行为疗法（R...",小心点总是没错的，但你也别忘了，你不是一个人在面对这些担心。跟信任的人聊一聊，哪怕只是说说心...
4,"[{'role': 'system', 'content': '你是一位精通理情行为疗法（R...",很好，等下次见面时，我们可以一起聊聊你发现的那些点滴。哪怕现在感觉前路模糊，也别忘了，你心里...
